(param-model-nb)=
# Parameter-dependent `TBModel`

In this example, we illustrate how to set hoppings and onsite terms symbolically to create a parameter-dependent tight-binding model. 

In [4]:
from pythtb import TBModel, Lattice
import numpy as np

Let's first make a finite model with two orbitals on a 1D chain. We will set:

- A parameterized onsite energy `"mA"` on orbital A
- A parameterized onsite energy `"mB"` on orbital B
- A parameterized hopping `"t"` between orbitals A and B

In [5]:
lat = Lattice(lat_vecs=[[1,]], orb_vecs=[[0],[1/2]])
model = TBModel(lattice=lat, spinful=False)

model.set_onsite("mA", ind_i=0)
model.set_onsite("mB", ind_i=1)
model.set_hop("t", 0, 1)
print(model)

----------------------------------------
       Tight-binding model report       
----------------------------------------
r-space dimension           = 1
k-space dimension           = 0
periodic directions         = []
spinful                     = False
number of spin components   = 1
number of electronic states = 2
number of orbitals          = 2

Lattice vectors (Cartesian):
  # 0 ===> [ 1.000]
Volume of unit cell (Cartesian) = 1.000 [A^d]

Orbital vectors (Cartesian):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
Orbital vectors (fractional):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
----------------------------------------
Site energies:
  # 0   ===> 'mA'
  # 1   ===> 'mB'
Hoppings:
  < 0 | H | 1 >  ===> 't'
Hopping distances:
  | pos( 0 ) - pos( 1 ) | =   0.500 (param)


We can utilize the parameterization to generate the Hamiltonian (and any other observables) at different parameter values without needing to redefine the model each time. To do this, we simply pass key word arguments to the relevant methods. We can either pass a single value or an array of values to evaluate the observable at multiple parameter points simultaneously.

In [6]:
model.parameters

[{'kind': 'onsite', 'orbitals': 0, 'names': ('mA',)},
 {'kind': 'onsite', 'orbitals': 1, 'names': ('mB',)},
 {'kind': 'hopping', 'orbitals': (0, 1), 'R': (0,), 'names': ('t',)}]

In [7]:
H = model.hamiltonian(mA = 0.1, mB = -0.1, t = np.linspace(0, 1, 10))  # Hamiltonian with t varying from 0 to 1, mA=0.1, mB=-0.1
print(f"H shape: {H.shape}")  # Should be (10, 2, 2) since there are 2 orbitals and 10 t values

H shape: (10, 2, 2)


Passing values to observables will not resolve the parameters in the model itself; it only affects the output of the observable being called. The model retains its symbolic parameters until they are explicitly set to numerical values using the `set_parameters` method.

In [8]:
print(f"Model parameters before setting: {model.parameters}")
model.set_parameters(mA=0.1, mB=-0.1, t=0.5)
print(f"Model parameters after setting: {model.parameters}")

print(model)

Model parameters before setting: [{'kind': 'onsite', 'orbitals': 0, 'names': ('mA',)}, {'kind': 'onsite', 'orbitals': 1, 'names': ('mB',)}, {'kind': 'hopping', 'orbitals': (0, 1), 'R': (0,), 'names': ('t',)}]
Model parameters after setting: []
----------------------------------------
       Tight-binding model report       
----------------------------------------
r-space dimension           = 1
k-space dimension           = 0
periodic directions         = []
spinful                     = False
number of spin components   = 1
number of electronic states = 2
number of orbitals          = 2

Lattice vectors (Cartesian):
  # 0 ===> [ 1.000]
Volume of unit cell (Cartesian) = 1.000 [A^d]

Orbital vectors (Cartesian):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
Orbital vectors (fractional):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
----------------------------------------
Site energies:
  # 0   ===>  0.100 
  # 1   ===> -0.100 
Hoppings:
  < 0 | H | 1 >  ===> 0.5000+0.0000j
Hopping distances:
  | 

## Functionally dependent parameters

We can also define parameters that are callables. These will take a parameter, and return the hopping or onsite value. This allows us to define parameters that depend on other parameters. A convenient way of doing this is using Python's `lambda` functions. 

In [9]:
lat = Lattice(lat_vecs=[[1,]], orb_vecs=[[0],[1/2]])
model = TBModel(lattice=lat, spinful=False)

model.set_onsite(lambda mA: np.cos(mA), ind_i=0)
model.set_onsite(lambda mB: np.sin(mB), ind_i=1)
model.set_hop(lambda mA, mB: (mA + mB) / 2, 0, 1)
print(model)

----------------------------------------
       Tight-binding model report       
----------------------------------------
r-space dimension           = 1
k-space dimension           = 0
periodic directions         = []
spinful                     = False
number of spin components   = 1
number of electronic states = 2
number of orbitals          = 2

Lattice vectors (Cartesian):
  # 0 ===> [ 1.000]
Volume of unit cell (Cartesian) = 1.000 [A^d]

Orbital vectors (Cartesian):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
Orbital vectors (fractional):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
----------------------------------------
Site energies:
  # 0   ===> model.set_onsite(lambda mA: np.cos(mA), ind_i=0)
  # 1   ===> model.set_onsite(lambda mB: np.sin(mB), ind_i=1)
Hoppings:
  < 0 | H | 1 >  ===> model.set_hop(lambda mA, mB: (mA + mB) / 2, 0, 1)
Hopping distances:
  | pos( 0 ) - pos( 1 ) | =   0.500 (param)


In [10]:
model.parameters


[{'kind': 'onsite',
  'orbitals': 0,
  'names': ('mA',),
  'source': 'model.set_onsite(lambda mA: np.cos(mA), ind_i=0)',
  'function': <function __main__.<lambda>(mA)>},
 {'kind': 'onsite',
  'orbitals': 1,
  'names': ('mB',),
  'source': 'model.set_onsite(lambda mB: np.sin(mB), ind_i=1)',
  'function': <function __main__.<lambda>(mB)>},
 {'kind': 'hopping',
  'orbitals': (0, 1),
  'R': (0,),
  'names': ('mA', 'mB'),
  'source': 'model.set_hop(lambda mA, mB: (mA + mB) / 2, 0, 1)',
  'function': <function __main__.<lambda>(mA, mB)>}]

We then resolve the parameters in the same way as before.

In [11]:
H = model.hamiltonian(mA = np.linspace(0, np.pi, 10), mB = np.linspace(0, np.pi, 12))  # Hamiltonian with mA and mB varying
print(f"H shape: {H.shape}")  # Should be (10, 12, 2, 2) since there are 2 orbitals, 10 mA values and 12 mB values

H shape: (10, 12, 2, 2)


In [12]:
model.set_parameters(mA=np.pi/4, mB=np.pi/4)
print(model)

----------------------------------------
       Tight-binding model report       
----------------------------------------
r-space dimension           = 1
k-space dimension           = 0
periodic directions         = []
spinful                     = False
number of spin components   = 1
number of electronic states = 2
number of orbitals          = 2

Lattice vectors (Cartesian):
  # 0 ===> [ 1.000]
Volume of unit cell (Cartesian) = 1.000 [A^d]

Orbital vectors (Cartesian):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
Orbital vectors (fractional):
  # 0 ===> [ 0.000]
  # 1 ===> [ 0.500]
----------------------------------------
Site energies:
  # 0   ===>  0.707 
  # 1   ===>  0.707 
Hoppings:
  < 0 | H | 1 >  ===> 0.7854+0.0000j
Hopping distances:
  | pos( 0 ) - pos( 1 ) | =   0.500
